# Inpactor3-SegNT · Entrenamiento en Colab

Genome Language Model (Nucleotide Transformer) + cabeza de segmentación
para detectar LTR-RTs en Arabidopsis.

**Antes de correr**: `Runtime → Change runtime type → T4 GPU`.

Tiempo total estimado: **~30-40 min** en GPU T4.

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Clonar repo e instalar dependencias

In [ ]:
%cd /content
!git clone https://github.com/Inpactor3/Inpactor3.git
%cd Inpactor3/Inpactor3_segnt
!pip install -q -r requirements.txt

## 3. Descargar TAIR10 y ground truth Inpactor2

El `Inpactor2_predictions.tab` sobre Arabidopsis debe estar disponible.
Si no lo tienes, puedes copiarlo del proyecto anterior de Inpactor3 o
correr Inpactor2 sobre TAIR10 primero.

In [ ]:
!bash scripts/download_arabidopsis.sh
!ls -lh data/raw/

In [ ]:
# Si tienes el ground truth localmente, súbelo con files.upload():
from google.colab import files
print('Sube Inpactor2_predictions.tab:')
uploaded = files.upload()
!mv Inpactor2_predictions.tab data/raw/ 2>/dev/null || true
!ls data/raw/

In [ ]:
!python scripts/build_labels.py \
    --genome data/raw/TAIR10.fasta \
    --annotations data/raw/Inpactor2_predictions.tab

## 4. Entrenar SegNT (10 épocas, ~20-30 min GPU T4)

In [ ]:
!python -m inpactor3_segnt.train --config configs/nt_50m.yaml

## 5. Inferencia sobre cromosoma 5 (examen)

In [ ]:
!python -m inpactor3_segnt.predict \
    --checkpoint models/best.pt \
    --genome data/raw/TAIR10.fasta \
    --scaffold 5 \
    --out results/chr5_predictions.tab

## 6. Evaluar contra Inpactor2 (métrica del informe)

In [ ]:
!python -m inpactor3_segnt.evaluate \
    --pred results/chr5_predictions.tab \
    --truth data/raw/Inpactor2_predictions.tab \
    --scaffold 5 \
    --genome data/raw/TAIR10.fasta \
    --report results/chr5_report.md

In [ ]:
!cat results/chr5_report.md

## 7. Descargar checkpoint entrenado a tu PC

In [ ]:
from google.colab import files
files.download('models/best.pt')
files.download('results/chr5_report.md')
files.download('results/chr5_predictions.tab')